# Preconditioned GAMERA/ReMIX Run From Scratch

This notebook is intentionally self-contained. It does not call the project scripts in `gamera_ireland/scripts`.

It does the complete workflow that produced the useful 3-hour video:

1. locate installed KaiPy and local Kaiju executables,
2. create a clean run directory,
3. generate the LFM grid using installed KaiPy,
4. write a constant southward-IMF `bcwind.h5`,
5. write a shallow GAMERA+ReMIX XML deck,
6. run Voltron for 3 hours with 1-minute output and 10-minute restarts,
7. monitor progress,
8. render FAC/Hall-current frames,
9. create the MP4 video.

This is a preconditioning run. The point is to let the magnetosphere settle before using restarts for case studies with changed solar wind.

## 0. Configuration

Run this notebook with the `SECSpy` environment. The run will take hours. Use a fresh `RUNID` each time; do not rerun into an old directory unless you deliberately delete it first.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.time import Time

import kaipy

PROJECT = Path('/home/simon/gits/SECpy_cleaned')
GI = PROJECT / 'gamera_ireland'
KAIJU = GI / 'external' / 'kaiju-master'
VOLTRON = KAIJU / 'build_serial' / 'bin' / 'voltron.x'
KAIPY_DIR = Path(kaipy.__file__).resolve().parent
GEN_LFM = KAIPY_DIR / 'scripts' / 'preproc' / 'genLFM.py'

RUNID = 'precon_southward_bz10_3h_clean'
RUN_DIR = GI / 'simulations' / RUNID
FRAMES_DIR = GI / 'output' / 'figures' / f'{RUNID}_movie'
VIDEO_PATH = FRAMES_DIR / f'{RUNID}_fac_hall.mp4'

T_FINAL_S = 10800.0
T_SPIN_S = 60.0
DT_OUTPUT_S = 60.0
DT_RESTART_S = 600.0
LOW_LAT_BOUNDARY_DEG = 45.0
REMIX_NP = 360
REMIX_NT = 45

print('Python:', sys.executable)
print('KaiPy:', kaipy.__file__)
print('genLFM:', GEN_LFM)
print('Voltron:', VOLTRON)
print('Run dir:', RUN_DIR)

In [ ]:
assert GEN_LFM.exists(), GEN_LFM
assert VOLTRON.exists(), VOLTRON
assert KAIJU.exists(), KAIJU

## 1. Helpers

These helpers live in the notebook. They are not imported from the project scripts.

The first group is about reproducibility and model setup: shell execution, clean run directories, solar-wind input, XML generation, restart discovery and progress checks. The comments are deliberately close to the code because these are the assumptions we will later turn into a proper workflow module.

In [ ]:
def run(cmd, cwd=None, env=None, check=True):
    """Print and execute a subprocess command.

    Keeping command execution behind one tiny helper makes the notebook easier
    to audit: every external command is printed before it runs, and failures
    raise immediately unless `check=False` is passed.
    """
    print('+', ' '.join(str(x) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=cwd, env=env, check=check)


def fresh_run_dir(path: Path):
    """Create a new run directory and refuse to overwrite old HDF5 output.

    GAMERA/ReMIX appends named groups such as `Step#0`, so rerunning in an old
    directory can produce confusing HDF5 `name already exists` diagnostics.
    A fresh directory is the safest default for reproducible runs.
    """
    if path.exists():
        raise FileExistsError(f'{path} already exists. Pick a new RUNID or delete it deliberately.')
    path.mkdir(parents=True)


def write_bcwind(path: Path):
    """Write the upstream solar-wind driver expected by Kaiju/GAMERA.

    This preconditioning driver is intentionally simple and constant: density
    5 cm^-3, Vx -450 km/s, By 0 nT, Bz -10 nT. The goal is not an event yet,
    but a settled southward-IMF magnetosphere that can later be restarted.
    """
    start = pd.Timestamp('1998-04-23T18:00:00')
    t_s = np.arange(0.0, T_FINAL_S + DT_OUTPUT_S, DT_OUTPUT_S)
    n = np.full(t_s.size, 5.0)                  # cm^-3
    temp = np.full(t_s.size, 1.0e5)              # K
    vx = np.full(t_s.size, -450.0e3)             # m/s
    vy = np.zeros(t_s.size)
    vz = np.zeros(t_s.size)
    bx = np.zeros(t_s.size)
    by = np.zeros(t_s.size)
    bz = np.full(t_s.size, -10.0)                # nT
    zeros = np.zeros(t_s.size)
    mjd0 = Time(start.to_pydatetime()).mjd
    mjd = mjd0 + t_s / 86400.0
    ut = np.asarray([(start + pd.Timedelta(seconds=float(t))).isoformat().encode('utf-8') for t in t_s])

    # Kaiju expects these dataset names and units for the solar-wind HDF5 file.
    # Velocity is in m/s, magnetic field is in nT, density is in cm^-3.
    with h5py.File(path, 'w') as h:
        h.attrs['source'] = 'Notebook-generated constant southward IMF preconditioning driver'
        h.attrs['start_utc'] = start.isoformat()
        for name, values in {
            'T': t_s, 'UT': ut, 'MJD': mjd,
            'D': n, 'Temp': temp,
            'Vx': vx, 'Vy': vy, 'Vz': vz,
            'Bx': bx, 'By': by, 'Bz': bz,
            'tilt': zeros, 'ae': zeros, 'al': zeros, 'au': zeros, 'symh': zeros,
            'Interped': np.zeros(t_s.size, dtype=np.int8),
            'f10.7': np.full(t_s.size, 120.0),
            'Kp': np.full(t_s.size, 4.0),
        }.items():
            h.create_dataset(name, data=values)
        h.create_dataset('Bx0', data=0.0)
        h.create_dataset('ByC', data=0.0)
        h.create_dataset('BzC', data=0.0)


def write_xml(path: Path):
    """Write the Kaiju XML deck for the validated shallow GAMERA+ReMIX run.

    `doDeep="F"` disables RAIJU/deep coupling. In this mode Kaiju requires the
    ReMIX precipitation model to be `FEDDER` or `SUNNY`; `LINMRG` is only valid
    for deep/RAIJU coupling. Restarts are written every 600 s so the final
    3-hour state is available as `Res.00018`.
    """
    text = f'''<Kaiju>
  <Gamera>
    <sim runid="{RUNID}" doH5g="T" H5Grid="lfmD.h5" icType="user" pdmb="0.75" rmeth="8C" />
    <floors dFloor="1.0e-4" pFloor="1.0e-6" />
    <timestep doCPR="T" limCPR="0.20" />
    <restart doRes="F" resID="{RUNID}" nRes="-1" />
    <physics doMHD="T" doBoris="T" Ca="20.0" />
    <ring gid="lfm" doRing="T" />
    <wind tsfile="bcwind.h5" />
    <source doSource="T" doWolfLim="T" doBounceDT="T" nBounce="1.0" />
    <iPdir N="1" bcPeriodic="F" />
    <jPdir N="1" bcPeriodic="F" />
    <kPdir N="1" bcPeriodic="T" />
    <coupling blockHalo="F" />
  </Gamera>
  <VOLTRON>
    <time tFin="{T_FINAL_S:.1f}" />
    <spinup doSpin="T" tSpin="{T_SPIN_S:.1f}" />
    <output dtOut="{DT_OUTPUT_S:.1f}" dtCon="30.0" />
    <coupling doGCM="F" dtCouple="5.0" doDeep="F" doAsyncCoupling="F" />
    <restart dtRes="{DT_RESTART_S:.1f}" />
    <imag doInit="T" />
    <helpers numHelpers="0" useHelpers="F" doSquishHelp="F" />
  </VOLTRON>
  <REMIX>
    <grid Np="{REMIX_NP}" Nt="{REMIX_NT}" LowLatBoundary="{LOW_LAT_BOUNDARY_DEG:.1f}" />
    <conductance doStarlight="T" apply_cap="T" const_sigma="F" />
    <precipitation aurora_model_type="FEDDER" beta="0.2" doAuroralSmooth="F" />
  </REMIX>
</Kaiju>
'''
    path.write_text(text)


def numeric_restarts(run_dir: Path, suffix: str):
    """Return numeric restart files for one component, sorted by restart ID.

    Kaiju may also leave `Res.XXXXX` files or links around. This helper ignores
    those placeholders and keeps only concrete restart numbers such as 00018.
    """
    out = []
    for p in run_dir.glob(f'{RUNID}.{suffix}.Res.*.h5'):
        try:
            n = int(p.name.split('.Res.')[1].split('.h5')[0])
        except Exception:
            continue
        out.append((n, p))
    return sorted(out)


def remix_steps(mix_path: Path):
    """List ReMIX output groups sorted by their integer step number."""
    if not mix_path.exists():
        return []
    with h5py.File(mix_path, 'r') as h:
        return sorted([k for k in h if k.startswith('Step#')], key=lambda s: int(s.split('#')[1]))


def status(run_dir: Path = RUN_DIR):
    """Print a compact progress summary from the ReMIX HDF5 output.

    This is safer than relying on Voltron stdout, which can be heavily buffered
    during long runs. Completion means the latest time reaches 10800 s.
    """
    mix = run_dir / f'{RUNID}.mix.h5'
    steps = remix_steps(mix)
    if not steps:
        print('No ReMIX steps yet')
        return None
    with h5py.File(mix, 'r') as h:
        g = h[steps[-1]]
        info = dict(step=steps[-1], nsteps=len(steps), time_s=float(g.attrs['time']), nCPCP=float(g.attrs['nCPCP']), sCPCP=float(g.attrs['sCPCP']))
    frac = info['time_s'] / T_FINAL_S
    restarts = numeric_restarts(run_dir, 'mix')
    print(f"{100*frac:5.1f}%  {info['step']}  t={info['time_s']:.1f}/{T_FINAL_S:.0f}s  CPCP={info['nCPCP']:.1f}/{info['sCPCP']:.1f} kV  restarts={len(restarts)}")
    if restarts:
        print('latest restart:', restarts[-1][1].name)
    return info

## 2. Create the run inputs

This creates a new directory, writes `lfmD.h5`, `bcwind.h5`, and the XML input deck.

In [ ]:
fresh_run_dir(RUN_DIR)

env = os.environ.copy()
env['KAIJUHOME'] = str(KAIJU)
env.setdefault('MPLCONFIGDIR', str(GI / 'environment' / 'matplotlib'))
env.setdefault('SPACEPY', str(GI / 'environment' / 'spacepy'))

run([sys.executable, GEN_LFM, '-gid', 'D'], cwd=RUN_DIR, env=env)
write_bcwind(RUN_DIR / 'bcwind.h5')
write_xml(RUN_DIR / f'{RUNID}.xml')

manifest = {
    'runid': RUNID,
    'run_dir': str(RUN_DIR),
    'mode': 'shallow GAMERA+ReMIX, no RAIJU',
    'driver': {'Vx_km_s': -450.0, 'n_cm3': 5.0, 'By_nT': 0.0, 'Bz_nT': -10.0},
    'duration_s': T_FINAL_S,
    'dt_output_s': DT_OUTPUT_S,
    'dt_restart_s': DT_RESTART_S,
}
(RUN_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
print(json.dumps(manifest, indent=2))

## 3. Run Voltron

This is the long step. It is written as a foreground run so the notebook can be run top-to-bottom. If you prefer a detached terminal run, copy the printed command and run it in a terminal.

In [ ]:
voltron_cmd = [VOLTRON, f'{RUNID}.xml']
print('cd', RUN_DIR)
print('KAIJUHOME=' + str(KAIJU), ' '.join(str(x) for x in voltron_cmd))

In [ ]:
# Long-running cell. Expected wall time was about 3.5 hours on this machine for the validated run.
# Comment this out if you launch Voltron in a terminal instead.
run(voltron_cmd, cwd=RUN_DIR, env=env)

## 4. Monitor progress

If the run is happening in another terminal, run this cell occasionally. A complete run should reach `Step#181` and `time_s >= 10800`.

In [ ]:
status()

In [ ]:
# Optional lightweight watch loop. Stop it with the notebook stop button.
# while True:
#     status()
#     time.sleep(60)

## 5. Plotting helpers

This directly reads ReMIX output. It does not do a SECS/Lompe inversion. The horizontal current shown is the physical Hall-current diagnostic derived from ReMIX potential and Hall conductance.

The MLT orientation and current-sign convention are the two easy places to make a scientifically misleading plot, so those assumptions are written directly into the helper docstrings.

In [ ]:
IONO_RADIUS_M = 6500.0e3


def load_remix_step(mix_h5: Path, step: str):
    """Load the ReMIX datasets needed for direct FAC/Hall diagnostics.

    The dataset names here come from the actual ReMIX HDF5 file produced by
    Kaiju. Potential is stored in kV and converted to volts before taking
    gradients. FAC is already in microampere per square metre in this output.
    """
    with h5py.File(mix_h5, 'r') as h:
        g = h[step]
        return {
            'timestamp': Time(float(g.attrs['MJD']), format='mjd').isot,
            'time_s': float(g.attrs['time']),
            'nCPCP': float(g.attrs['nCPCP']),
            'sCPCP': float(g.attrs['sCPCP']),
            'potential_v': g['Potential NORTH'][...] * 1e3,
            'fac': g['Field-aligned current NORTH'][...],
            'hall_s': g['Hall conductance NORTH'][...],
        }


def native_grid(shape, low_lat_boundary_deg=LOW_LAT_BOUNDARY_DEG):
    """Construct the native ReMIX polar plotting grid.

    ReMIX stores the northern ionosphere as colatitude/theta by longitude/phi.
    The plot convention used here is the standard MLT polar layout:
    12 MLT at the top, 06 at right, 00 at bottom, and 18 at left.
    """
    ntheta, nphi = shape
    dtheta = np.deg2rad(low_lat_boundary_deg / ntheta)
    theta = dtheta * (np.arange(ntheta) + 1.0)
    phi = 2*np.pi*np.arange(nphi)/nphi
    phig, thetag = np.meshgrid(phi, theta)
    theta_edges = np.empty(ntheta + 1)
    theta_edges[0] = 0.0
    theta_edges[1:-1] = 0.5*(theta[:-1] + theta[1:])
    theta_edges[-1] = theta[-1]
    phi_edges = 2*np.pi*np.arange(nphi + 1)/nphi
    peg, teg = np.meshgrid(phi_edges, theta_edges)
    # The minus sign puts dusk/18 MLT on the left and dawn/06 MLT on the right.
    x = -np.rad2deg(thetag)*np.sin(phig)
    y = np.rad2deg(thetag)*np.cos(phig)
    xe = -np.rad2deg(teg)*np.sin(peg)
    ye = np.rad2deg(teg)*np.cos(peg)
    return theta, phi, thetag, phig, x, y, xe, ye, np.rad2deg(dtheta)


def hall_current(remix, grid):
    """Derive horizontal Hall current from potential and Hall conductance.

    This is a direct ReMIX diagnostic, not a SECS/Lompe inversion. We compute
    E = -grad(Phi) on the spherical ionospheric shell, then use the same
    northern-hemisphere Hall-current sign convention used by Kaiju/calcdb:
    J_theta = -Sigma_H E_phi / nhcdip and J_phi = Sigma_H E_theta / nhcdip.
    """
    theta, phi, thetag, phig, *_ = grid
    pot = np.asarray(remix['potential_v'], float)
    hall = np.asarray(remix['hall_s'], float)
    # Gradients on a spherical shell: theta derivative is meridional, phi
    # derivative is periodic and divided by sin(theta) below.
    dpot_dtheta = np.gradient(pot, theta, axis=0, edge_order=2)
    dphi = phi[1] - phi[0]
    dpot_dphi = (np.roll(pot, -1, axis=1) - np.roll(pot, 1, axis=1))/(2*dphi)
    e_theta = -dpot_dtheta / IONO_RADIUS_M
    e_phi = -dpot_dphi / (IONO_RADIUS_M * np.maximum(np.sin(thetag), np.sin(np.deg2rad(1.0))))
    # Dip-angle denominator from the northern-hemisphere calcdb convention.
    nhcdip = -2*np.cos(thetag)/np.sqrt(1 + 3*np.cos(thetag)**2)
    jtheta = -hall * e_phi / nhcdip
    jphi = hall * e_theta / nhcdip
    jx = -jtheta*np.sin(phig) - jphi*np.cos(phig)
    jy = jtheta*np.cos(phig) - jphi*np.sin(phig)
    return jtheta, jphi, jx, jy, np.hypot(jtheta, jphi)


def add_mlt_guides(ax, low_lat_boundary_deg=LOW_LAT_BOUNDARY_DEG):
    """Add magnetic-latitude rings and the 00/06/12/18 MLT spokes."""
    phi = np.linspace(0, 2*np.pi, 361)
    for mlat in [80, 70, 60, 50]:
        r = 90 - mlat
        if r <= low_lat_boundary_deg:
            ax.plot(r*np.sin(phi), r*np.cos(phi), ':', color='0.55', lw=0.6)
            ax.text(0, r, str(mlat), ha='left', va='bottom', fontsize=8, color='0.35')
    for mlt, angle in [(12, 0), (18, np.pi/2), (0, np.pi), (6, 3*np.pi/2)]:
        ax.plot([0, -low_lat_boundary_deg*np.sin(angle)], [0, low_lat_boundary_deg*np.cos(angle)], ':', color='0.65', lw=0.6)
        ax.text(-1.05*low_lat_boundary_deg*np.sin(angle), 1.05*low_lat_boundary_deg*np.cos(angle), f'{mlt:02d}', ha='center', va='center', fontsize=9, color='0.25')


def plot_frame(mix_h5: Path, step: str, output: Path, theta_stride=2, phi_stride=8, quiver_length_deg=1.15):
    """Render one two-panel diagnostic frame.

    Left: field-aligned current with convection-potential contours.
    Right: signed azimuthal Hall current as colour, with horizontal Hall
    current vectors as quivers. Quivers are masked very near the pole because
    the native polar coordinate basis is singular there.
    """
    r = load_remix_step(mix_h5, step)
    grid = native_grid(r['fac'].shape)
    theta, phi, thetag, phig, x, y, xe, ye, dtheta_deg = grid
    jtheta, jphi, jx, jy, jmag = hall_current(r, grid)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.8), constrained_layout=True)
    fac_lim = max(0.25, float(np.nanpercentile(np.abs(r['fac']), 99)))
    m0 = axes[0].pcolormesh(xe, ye, r['fac'], cmap='seismic', vmin=-fac_lim, vmax=fac_lim, shading='flat')
    axes[0].contour(x, y, r['potential_v']*1e-3, levels=np.r_[-200:201:10], colors='black', linewidths=0.55)
    fig.colorbar(m0, ax=axes[0], label='FAC [$\\mu$A m$^{-2}$]')
    axes[0].set_title('Field-aligned current')
    jphi_ma = jphi * 1e3
    j_lim = max(20, float(np.nanpercentile(np.abs(jphi_ma), 99)))
    m1 = axes[1].pcolormesh(xe, ye, jphi_ma, cmap='seismic', vmin=-j_lim, vmax=j_lim, shading='flat')
    mag95 = max(float(np.nanpercentile(jmag, 95)), 1e-12)
    gain = quiver_length_deg*np.clip(jmag/mag95, 0, 1.6)
    norm = np.maximum(jmag, 1e-12)
    qx = jx/norm*gain
    qy = jy/norm*gain
    # Avoid drawing misleading vectors at the polar coordinate singularity.
    qx = np.where(theta[:, None] < np.deg2rad(2.5*dtheta_deg), np.nan, qx)
    qy = np.where(theta[:, None] < np.deg2rad(2.5*dtheta_deg), np.nan, qy)
    skip = (slice(None, None, theta_stride), slice(None, None, phi_stride))
    axes[1].quiver(x[skip], y[skip], qx[skip], qy[skip], color='white', angles='xy', scale_units='xy', scale=1, width=0.0021, alpha=0.92)
    axes[1].contour(x, y, r['potential_v']*1e-3, levels=np.r_[-200:201:10], colors='white', linewidths=0.45, alpha=0.6)
    fig.colorbar(m1, ax=axes[1], label='Hall $J_\\phi$ [mA m$^{-1}$]')
    axes[1].set_title('Signed Hall electrojet')
    for ax in axes:
        add_mlt_guides(ax)
        ax.set_aspect('equal')
        ax.set_xlim(-LOW_LAT_BOUNDARY_DEG*1.12, LOW_LAT_BOUNDARY_DEG*1.12)
        ax.set_ylim(-LOW_LAT_BOUNDARY_DEG*1.12, LOW_LAT_BOUNDARY_DEG*1.12)
        ax.set_axis_off()
    fig.suptitle(f'{RUNID} {step}, {r["timestamp"]}  CPCP(N/S)={r["nCPCP"]:.1f}/{r["sCPCP"]:.1f} kV')
    output.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output, dpi=180)
    plt.close(fig)
    return {'step': step, 'time_s': r['time_s'], 'timestamp': r['timestamp'], 'nCPCP': r['nCPCP'], 'sCPCP': r['sCPCP']}

## 6. Render all one-minute frames

This skips `Step#0`, because that is the spinup state. It renders `Step#1` through the final step.

In [ ]:
mix_h5 = RUN_DIR / f'{RUNID}.mix.h5'
steps = [s for s in remix_steps(mix_h5) if int(s.split('#')[1]) >= 1]
print(len(steps), steps[0], steps[-1])

In [ ]:
FRAMES_DIR.mkdir(parents=True, exist_ok=True)
manifest = []
for i, step in enumerate(steps, start=1):
    frame = FRAMES_DIR / f'frame{i:04d}.png'
    item = plot_frame(mix_h5, step, frame)
    manifest.append({'frame': frame.name, **item})
    if i % 10 == 0 or i == 1 or i == len(steps):
        print(i, '/', len(steps), step)
(FRAMES_DIR / 'frames_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
print('frames:', FRAMES_DIR)

## 7. Build the MP4

This uses the system `ffmpeg` directly. At `fps=6`, a 181-frame, 3-hour run becomes about 30 seconds of video.

In [ ]:
ffmpeg_cmd = [
    'ffmpeg', '-hide_banner', '-stats', '-y',
    '-framerate', '6',
    '-start_number', '1',
    '-i', str(FRAMES_DIR / 'frame%04d.png'),
    '-vf', 'scale=1980:1044:flags=lanczos',
    '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '23', '-pix_fmt', 'yuv420p',
    str(VIDEO_PATH),
]
run(ffmpeg_cmd, cwd=PROJECT)
print(VIDEO_PATH)

## 8. Use the preconditioned restart for later case studies

The validated preconditioning run writes restarts every 600 s. After 3 hours the useful checkpoint is `Res.00018`.

Local Kaiju examples restart by changing the `<Gamera><restart ... /></Gamera>` element to:

```xml
<restart doRes="T" resID="precon_southward_bz10_3h_clean" nRes="18" />
```

The next smoke test should create a short follow-on run from this checkpoint with a changed `bcwind.h5`, then verify that `Step#1` begins from the developed convection state rather than from a cold start.

In [ ]:
print('Latest mix restart:', numeric_restarts(RUN_DIR, 'mix')[-1][1])
print('Latest GAMERA restart:', numeric_restarts(RUN_DIR, 'gam')[-1][1])
print('Video:', VIDEO_PATH)